# Calibracion del Sistema de Scoring

**Modulo:** Pattern Detection  
**Objetivo:** Entender el sistema de scoring y confianza  
**Duracion estimada:** 30 minutos

---

## Contenido

1. [Setup](#setup)
2. [Niveles de Confianza](#niveles-de-confianza)
3. [Componentes del Score](#componentes)
4. [Resolucion de Conflictos](#conflictos)
5. [Ejemplos Practicos](#ejemplos)
6. [Ejercicios](#ejercicios)

---

## 1. Setup

In [ ]:
# Agregar path del proyecto
import sys
sys.path.insert(0, '../..')

# Imports necesarios
from app.core.parser import parse_pseudocode
from app.core.patterns.pattern_detector import PatternDetector
from app.core.patterns.pattern_scorer import PatternScorer
from app.core.patterns.base_pattern import ConfidenceLevel

# Para visualizacion
import pandas as pd

# Crear instancias
detector = PatternDetector()
scorer = PatternScorer()

print("Setup completado")

---

## 2. Niveles de Confianza

| Nivel | Rango | Interpretacion |
|-------|-------|----------------|
| Muy Alto | >= 0.90 | Deteccion segura |
| Alto | 0.70 - 0.89 | Confiable |
| Medio | 0.50 - 0.69 | Probable |
| Bajo | 0.30 - 0.49 | Posible |
| Muy Bajo | < 0.30 | Dudosa |

### Ejemplo: Ver Niveles de Confianza

In [ ]:
# Ver todos los niveles de confianza
print("=== NIVELES DE CONFIANZA ===")
for level in ConfidenceLevel:
    print(f"  - {level.value}")

# Ejemplo con algoritmo de alta confianza
codigo_merge = """
algorithm mergeSort(A[], p, r)
begin
    if (p < r) then
        q <- floor((p + r) / 2)
        call mergeSort(A, p, q)
        call mergeSort(A, q + 1, r)
        call merge(A, p, q, r)
    end
end
"""

ast = parse_pseudocode(codigo_merge)
resultado = detector.detect(ast)

primary = resultado.all_patterns[0] if resultado.all_patterns else None
if primary:
    print(f"\n=== MERGE SORT ===")
    print(f"Score: {primary.final_score:.2%}")
    print(f"Nivel: {primary.pattern.confidence_level.value}")

---

## 3. Componentes del Score

El score se calcula considerando:

| Factor | Peso | Descripcion |
|--------|------|-------------|
| ast_structure | 35% | Estructura del AST |
| code_keywords | 25% | Palabras clave |
| complexity_match | 20% | Coincidencia con complejidad |
| confidence | 20% | Confianza del detector |

### Ejemplo: Comparar Scores

In [ ]:
# Comparar scores de diferentes algoritmos
algoritmos = {
    "Merge Sort": codigo_merge,
    "Bubble Sort": """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
""",
    "Fibonacci Recursivo": """
algorithm fibonacci(n)
begin
    if (n <= 1) then
        return n
    end
    return call fibonacci(n - 1) + call fibonacci(n - 2)
end
"""
}

resultados = []
for nombre, codigo in algoritmos.items():
    ast = parse_pseudocode(codigo)
    resultado = detector.detect(ast)
    
    for sp in resultado.all_patterns[:3]:  # Top 3 patrones
        resultados.append({
            "Algoritmo": nombre,
            "Patron": sp.pattern.pattern_name,
            "Score": f"{sp.final_score:.0%}",
            "Nivel": sp.pattern.confidence_level.value
        })

df = pd.DataFrame(resultados)
print("=== SCORES POR ALGORITMO ===")
print(df.to_string(index=False))

---

## 4. Resolucion de Conflictos

Algunos patrones son mutuamente excluyentes:
- Fuerza Bruta vs Programacion Dinamica
- Greedy vs Backtracking

### Penalizaciones

| Conflicto | Penalizacion |
|-----------|--------------|
| Patron en conflicto | -15% |
| >50% indicadores faltantes | -10% |

### Ejemplo: Ver Conflictos

In [ ]:
# Ver conflictos en deteccion
codigo_quick = """
algorithm quickSort(A[], low, high)
begin
    if (low < high) then
        p <- call partition(A, low, high)
        call quickSort(A, low, p - 1)
        call quickSort(A, p + 1, high)
    end
end
"""

ast = parse_pseudocode(codigo_quick)
resultado = detector.detect(ast)

print("=== QUICK SORT: ANALISIS DE CONFLICTOS ===")
for sp in resultado.all_patterns:
    print(f"\n{sp.pattern.pattern_name}:")
    print(f"  Score: {sp.final_score:.0%}")
    print(f"  Es primario: {'Si' if sp.is_primary else 'No'}")
    if sp.conflicts:
        print(f"  Conflictos: {sp.conflicts}")

---

## 5. Ejercicios

### Ejercicio 1: Predecir el Score

In [ ]:
# Ejercicio 1: Que nivel de confianza tendra este algoritmo?
# Predice: muy_alto, alto, medio, bajo, muy_bajo

codigo_simple = """
algorithm simpleLoop(n)
begin
    for i <- 1 to n do
        x <- x + 1
    end
end
"""

ast = parse_pseudocode(codigo_simple)
resultado = detector.detect(ast)

print("Resultado:")
if resultado.all_patterns:
    primary = resultado.all_patterns[0]
    print(f"Patron: {primary.pattern.pattern_name}")
    print(f"Score: {primary.final_score:.0%}")
    print(f"Nivel: {primary.pattern.confidence_level.value}")
else:
    print("No se detectaron patrones con confianza suficiente")

### Ejercicio 2: Mejorar el Nivel de Confianza

Modifica el algoritmo para que alcance un nivel de confianza HIGH.

In [ ]:
# Ejercicio 2: Modifica este algoritmo para alcanzar HIGH confidence
# Pista: Agrega mas indicadores del patron (llamada recursiva, division, etc.)

# Algoritmo base (MEDIUM confidence)
code_base = """
function algoritmo(A, inicio, fin):
    if inicio >= fin:
        return A[inicio]
    
    medio = (inicio + fin) / 2
    resultado = algoritmo(A, inicio, medio) + algoritmo(A, medio+1, fin)
    return resultado
"""

# TU CODIGO: Mejora el algoritmo para que tenga HIGH confidence
code_mejorado = """
# Escribe aqui tu algoritmo mejorado
"""

# Verifica tu solucion
print("=== Algoritmo Base ===")
ast_base = parse_pseudocode(code_base)
pattern_base = detector.detect(ast_base)
print(f"Score: {pattern_base['score']:.2f}")
print(f"Confianza: {pattern_base['confidence'].value}")

print("\n=== Tu Algoritmo Mejorado ===")
# Descomenta cuando hayas escrito tu algoritmo
# ast_mejorado = parse_pseudocode(code_mejorado)
# pattern_mejorado = detector.detect(ast_mejorado)
# print(f"Score: {pattern_mejorado['score']:.2f}")
# print(f"Confianza: {pattern_mejorado['confidence'].value}")

## 6. Tips y Resumen

### 💡 Tips

- **Score Alto ≠ Certeza Absoluta**: Un score de 0.9 indica alta probabilidad, no garantia
- **Multiples Patrones**: Es normal que un algoritmo tenga indicadores de varios patrones
- **Contexto Importa**: El mismo indicador puede pesar diferente segun el contexto
- **Calibracion Iterativa**: Ajusta umbrales segun los falsos positivos/negativos observados
- **Indicadores Clave**: Enfocate en los indicadores mas distintivos de cada patron

### 📝 Resumen

En este notebook aprendimos:

1. **Niveles de Confianza**: LOW (<0.5), MEDIUM (0.5-0.8), HIGH (>0.8)
2. **Componentes del Score**: Como diferentes indicadores contribuyen al score total
3. **Comparacion de Algoritmos**: Visualizar scores de multiples algoritmos
4. **Resolucion de Conflictos**: Manejar cuando un algoritmo muestra multiples patrones
5. **Calibracion Practica**: Ajustar expectativas basadas en resultados

### 🔗 Proximos Pasos

- Explorar [pattern_evaluation.ipynb](pattern_evaluation.ipynb) para evaluacion avanzada
- Continuar con el modulo [04_data_structures](../04_data_structures/) para deteccion de estructuras